# Model Inference

After successfully training the De Novo model, you can use it to predict peptide sequences from new mass spectrometry data.

Below is a detailed introduction to the parameters in the inference.yaml configuration file:

## Global & Path Settings

-   `mode`: Defines the operational mode of the script.
    -   "eval": Use this if your dataset contains ground-truth labels. The labels will be saved in the output csv file.
    -   "denovo": Use this if your dataset consists of purely unlabeled raw spectra.

-   `device`: Specifies the hardware device for inference.
-   `task_name`: An identifier for the current inference job (e.g., "ricebean"). This is used to name the output result files.
-   `output_dir`: The directory where the final prediction results will be saved.
-   `checkpoint_path`: The path to your fully trained De Novo model weight file (.ckpt).
-   `clip_checkpoint_path`: The path to your pre-trained CLIP model weight file.

## prediction (Decoding & Filtering Parameters)

This block controls the autoregressive generation process and post-prediction filtering.
-   `num_beams`: Controls the beam search decoding strategy.
    Setting it to 0 typically uses Greedy Search (selecting the single most probable amino acid at each step), which is extremely fast. Setting it to an integer $>= 1$ (e.g., 5) enables Beam Search, which explores multiple sequence paths simultaneously to find a higher-probability overall sequence, though it requires more computation time.
-   `min_len` / `max_len`: The length constraints for the generated peptide sequences. The model will stop generating if it hits max_len and will penalize or discard sequences shorter than min_len (e.g., 6 to 100 amino acids).
-   `max_isotope`: The maximum isotope shift allowed when calculating the precursor mass difference.
-   `mass_tolerance`: The acceptable mass error threshold (typically in ppm, e.g., 50.0).
-   `gradscaling_enabled`: Enables automated mixed precision (AMP) or gradient scaling during inference.

## data (Dataset Configuration)

-   `test_path`: The path to the preprocessed HDF5 test dataset.
-   `test_batch_size`: [Deprecated] This parameter is no longer used. The system now automatically makes a conservative estimate for the batch size internally to optimize GPU utilization while preventing out-of-memory (OOM) errors.
-   `n_workers`: The number of CPU worker processes used for data loading.

In [1]:
from ruamel.yaml import YAML

# replace the data path with your own path

yaml = YAML()
yaml.preserve_quotes = True
yaml.indent(mapping=2, sequence=4, offset=2)

config = """
mode: "eval"
device: "cuda:0"
task_name: "ricebean"
output_dir: "./outputs/final/results"
checkpoint_path: "./outputs/checkpoints/denovo/checkpoints_step=718178_val_total_accuracy=0.8394.ckpt"
clip_checkpoint_path: "./outputs/checkpoints/clip/checkpoints_step=467635_val_top1_accuracy=0.9696.ckpt"

prediction:
  num_beams: 0
  min_len: 6
  max_len: 100
  max_isotope: 1
  mass_tolerance: 50.0
  gradscaling_enabled: true

data:
  test_path: ""
  test_batch_size: 512
  n_workers: 16
"""

config = yaml.load(config)
yaml.dump(config, open("/data2/xp/RocNovo-Lightning/outputs/inference.yaml", "w"))

In [ ]:
import sys
sys.path.append("..")

from rocnovo.inference.inference import predict

predict("/data2/xp/RocNovo-Lightning/outputs/inference.yaml")